# Final model comparison

This notebook consolidates final temporal-test performance. It does not retrain models or select features.

In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
from pathlib import Path
import sys
import yaml
import joblib
import pandas as pd
from IPython.display import display

CONFIG_PATH = Path('../src/icare_risk/config/icare/analysis.yaml')
with CONFIG_PATH.open() as f:
    config = yaml.safe_load(f)
from icare_risk.data.audit import load_admission_file
from icare_risk.evaluation.final_comparison import (
    evaluate_models_on_table, add_feature_metadata, rank_models, save_final_comparison,
)

In [4]:
prediction_path = Path(config['outputs']['prediction_directory']) / 'admission_predictions.parquet'
if not prediction_path.exists():
    prediction_path = Path(config['outputs']['prediction_directory']) / 'admission_predictions.csv'
predictions = load_admission_file(prediction_path)
target = config['columns']['target']
target_mapping = config.get('target_mapping', config.get('analysis', {}).get('target_mapping', {}))
test = predictions[predictions['split'].eq('test')].copy()
prediction_columns = [c for c in test.columns if c.endswith('_prob')]
print(f'Test admissions: {len(test):,}')
print(f'Model prediction columns: {len(prediction_columns)}')

Test admissions: 2,483
Model prediction columns: 6


In [5]:
comparison = evaluate_models_on_table(
    test, target_column=target, prediction_columns=prediction_columns,
    target_mapping=target_mapping,
    threshold=config.get('analysis', {}).get('evaluation_threshold', 0.5),
)
model_configs = config.get('models', [])
feature_counts = {}
for model_cfg in model_configs:
    path = Path(config['outputs']['model_directory']) / f"{model_cfg['name']}.joblib"
    if path.exists():
        bundle = joblib.load(path)
        if isinstance(bundle, dict) and bundle.get('features') is not None:
            feature_counts[model_cfg['name']] = len(bundle['features'])
comparison = add_feature_metadata(comparison, model_configs=model_configs, feature_counts=feature_counts)
ranked = rank_models(comparison)
display(ranked)

ModuleNotFoundError: No module named 'xgboost'

In [5]:
output_dir = Path(config['outputs'].get('final_directory', 'results/final'))
output_dir.mkdir(parents=True, exist_ok=True)
ranked.to_csv(output_dir / 'final_model_comparison.csv', index=False)
ranked.to_csv(output_dir / 'final_model_ranking.csv', index=False)
print(f'Saved results to {output_dir.resolve()}')

Saved results to /app/notebooks/results/final
